# Aula 5 — Redes Recorrentes: RNNs e LSTMs (Keras/TensorFlow)

**CCM-109 — Deep Learning · Prof. Ronaldo Prati (UFABC)**

Este notebook acompanha a Aula 5 e cobre:

1. **Dataset IMDb** — carregamento e exploração
2. **SimpleRNN** — baseline recorrente básico
3. **LSTM** — memória de longo prazo
4. **LSTM Bidirecional Empilhado** — arquitetura completa
5. **GRU** — alternativa eficiente
6. **Comparativo** — curvas de aprendizado e métricas

## 0. Setup

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow {tf.__version__}  |  GPU: {len(tf.config.list_physical_devices('GPU'))} disponível(is)")

## 1. Dataset — IMDb

O `keras.datasets.imdb` fornece 50.000 críticas de filmes pré-tokenizadas como sequências de inteiros.
Cada inteiro representa uma palavra do vocabulário (ordenado por frequência).

In [ ]:
MAX_VOCAB = 10_000
MAX_LEN   = 200

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(
    num_words=MAX_VOCAB
)

print(f"Treino: {X_train.shape}  |  Teste: {X_test.shape}")
print(f"Comprimentos das sequências: min={min(len(x) for x in X_train)}, "
      f"max={max(len(x) for x in X_train)}, "
      f"mediana={np.median([len(x) for x in X_train]):.0f}")

# Distribuição de comprimentos
plt.figure(figsize=(8, 3))
plt.hist([len(x) for x in X_train], bins=50, color="steelblue", edgecolor="white")
plt.axvline(MAX_LEN, color="crimson", ls="--", label=f"MAX_LEN={MAX_LEN}")
plt.xlabel("Comprimento da sequência")
plt.ylabel("Frequência")
plt.title("Distribuição de comprimento das críticas (treino)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Decodificar uma crítica para ver o texto original
word_index = keras.datasets.imdb.get_word_index()
inv_index  = {v + 3: k for k, v in word_index.items()}
inv_index.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

def decode(seq):
    return " ".join(inv_index.get(i, "?") for i in seq)

print("Exemplo (label =", y_train[0], "= positivo):")
print(decode(X_train[0])[:400], "...")

In [ ]:
# Padding / truncamento para comprimento fixo
X_train_pad = keras.utils.pad_sequences(
    X_train, maxlen=MAX_LEN, padding="post", truncating="post"
)
X_test_pad  = keras.utils.pad_sequences(
    X_test,  maxlen=MAX_LEN, padding="post", truncating="post"
)
print(f"X_train_pad: {X_train_pad.shape}  dtype: {X_train_pad.dtype}")

## 2. Baseline: SimpleRNN

Vamos começar com a RNN vanilla para estabelecer um baseline.

In [ ]:
def build_simple_rnn(units=64, dropout=0.3):
    return keras.Sequential([
        keras.Input(shape=(MAX_LEN,)),
        keras.layers.Embedding(MAX_VOCAB, 128, mask_zero=True),
        keras.layers.SimpleRNN(units, dropout=dropout),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1, activation="sigmoid"),
    ], name="SimpleRNN")

model_rnn = build_simple_rnn()
model_rnn.summary()

In [ ]:
model_rnn.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(1e-3),
    metrics=["accuracy"]
)

hist_rnn = model_rnn.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

## 3. LSTM

A LSTM resolve o vanishing gradient com sua célula de memória e três gates (forget, input, output).
Vamos comparar com e sem `recurrent_dropout` (dropout variacional nos estados recorrentes).

In [ ]:
def build_lstm(units=128, dropout=0.2, recurrent_dropout=0.0):
    return keras.Sequential([
        keras.Input(shape=(MAX_LEN,)),
        keras.layers.Embedding(MAX_VOCAB, 128, mask_zero=True),
        keras.layers.LSTM(
            units,
            dropout=dropout,
            recurrent_dropout=recurrent_dropout  # dropout variacional
        ),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1, activation="sigmoid"),
    ], name="LSTM")

model_lstm = build_lstm(units=128, dropout=0.2)
model_lstm.summary()

In [ ]:
model_lstm.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(1e-3),
    metrics=["accuracy"]
)

hist_lstm = model_lstm.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

## 4. LSTM Bidirecional Empilhado

Arquitetura mais completa:
- **Bidirecional**: processa a sequência para frente e para trás, dobrando o vetor de saída
- **Empilhado**: a saída de cada camada alimenta a próxima (`return_sequences=True`)
- **`mask_zero=True`**: evita que os tokens de padding influenciem o treinamento

In [ ]:
def build_bilstm_stacked(units=128, dropout=0.2):
    return keras.Sequential([
        keras.Input(shape=(MAX_LEN,)),
        keras.layers.Embedding(MAX_VOCAB, 128, mask_zero=True),
        keras.layers.Bidirectional(
            keras.layers.LSTM(units, return_sequences=True,
                              dropout=dropout, recurrent_dropout=0.1)
        ),
        keras.layers.Bidirectional(
            keras.layers.LSTM(units // 2, dropout=dropout)
        ),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1, activation="sigmoid"),
    ], name="BiLSTM_stacked")

model_bilstm = build_bilstm_stacked()
model_bilstm.summary()

In [ ]:
model_bilstm.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(5e-4),
    metrics=["accuracy"]
)

hist_bilstm = model_bilstm.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

## 5. GRU — Alternativa Eficiente

O GRU tem menos parâmetros que a LSTM (2 gates em vez de 3) e frequentemente atinge desempenho comparável.

In [ ]:
def build_gru(units=128, dropout=0.2):
    return keras.Sequential([
        keras.Input(shape=(MAX_LEN,)),
        keras.layers.Embedding(MAX_VOCAB, 128, mask_zero=True),
        keras.layers.GRU(
            units,
            dropout=dropout,
            recurrent_dropout=0.1
        ),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1, activation="sigmoid"),
    ], name="GRU")

model_gru = build_gru()
model_gru.summary()

In [ ]:
model_gru.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(1e-3),
    metrics=["accuracy"]
)

hist_gru = model_gru.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

## 6. Comparativo

In [ ]:
models_info = {
    "SimpleRNN":    (model_rnn,    hist_rnn),
    "LSTM":         (model_lstm,   hist_lstm),
    "BiLSTM stack": (model_bilstm, hist_bilstm),
    "GRU":          (model_gru,    hist_gru),
}
colors = ["gray", "steelblue", "darkorange", "seagreen"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metric, title in zip(
    axes,
    ["val_loss", "val_accuracy"],
    ["Loss (validação)", "Acurácia (validação)"]
):
    for (name, (_, hist)), color in zip(models_info.items(), colors):
        vals = hist.history[metric]
        ax.plot(range(1, len(vals)+1), vals, label=name, color=color, lw=2, marker="o")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.legend()

plt.tight_layout()
plt.show()

# Avaliar no conjunto de teste completo
print(f"\n{'Modelo':<15} {'Val acc':>8} {'Test acc':>9} {'Params':>12}")
print("-" * 46)
for name, (model, hist) in models_info.items():
    best_val  = max(hist.history["val_accuracy"])
    _, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
    params    = model.count_params()
    print(f"{name:<15} {best_val:>8.3f} {test_acc:>9.3f} {params:>12,}")

## 7. Predição em Textos Novos

Vamos testar o modelo BiLSTM em críticas novas usando `TextVectorization`.

In [ ]:
# Reconstruir o modelo com TextVectorization para aceitar texto bruto
vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_sequence_length=MAX_LEN,
    standardize="lower_and_strip_punctuation"
)

# Adaptar ao texto de treino (precisamos decodificar de volta)
train_texts = [decode(x) for x in X_train[:5000]]   # amostra para velocidade
vectorizer.adapt(train_texts)

model_text = keras.Sequential([
    keras.Input(shape=(1,), dtype="string"),
    vectorizer,
    keras.layers.Embedding(MAX_VOCAB, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(128, dropout=0.2)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid"),
])
model_text.compile(loss="binary_crossentropy",
                   optimizer=keras.optimizers.Adam(5e-4),
                   metrics=["accuracy"])

print("Modelo com TextVectorization pronto para texto bruto")
model_text.summary()

In [ ]:
examples = [
    "This movie was absolutely fantastic! The acting was superb and the story was gripping.",
    "Terrible film. Boring plot, bad acting, complete waste of time.",
    "It was okay, nothing special but not terrible either.",
    "One of the best movies I have ever seen. Highly recommended!",
]

# Usar o modelo bilstm (já treinado com índices pré-computados)
print("Predições com BiLSTM (índices pré-computados):")
print("-" * 55)
for text in examples:
    tokens = keras.utils.pad_sequences(
        [[word_index.get(w.lower(), 2) for w in text.split()]],
        maxlen=MAX_LEN, padding="post"
    )
    prob = model_bilstm.predict(tokens, verbose=0)[0][0]
    label = "Positivo 😊" if prob > 0.5 else "Negativo 😞"
    print(f"[{prob:.2%}] {label}")
    print(f"  → {text[:70]}...")
    print()